# Notebook 26 — MJO Barlow Twins (τ-graded, D=7 NSV-matched)
**Project:** ENSO-BSISO SSL — MJO extension  
**Author:** Jiayi (jh9141@nyu.edu)

Self-supervised **Barlow Twins** (Zbontar et al. 2021) on the MJO intraseasonal field, with the project-specific **temporally-graded loss** decided in Session 36 of the conversation log.

## Method

Two views = a frame and a **τ-day-later** frame `(X_t, X_{t+τ})`, τ ∈ {1,2,3,4,5}. Encoder `f` (warm-started from the MJO NSV Stage-1 encoder) → 64-D, then a small projector → **D=7** (matching the MJO NSV intrinsic dimension d̂=7). On the 7×7 cross-correlation matrix `C(τ)`:

$$\mathcal{L}(\tau) = \lambda_\text{inv}(\tau)\sum_i (1 - C_{ii}(\tau))^2 \; + \; \lambda_\text{off}(\tau)\sum_i\sum_{j\neq i} C_{ij}(\tau)^2$$

**Both** terms are τ-graded (Session-36 decision #1): `λ(τ)` decays linearly 1.0→0.5 as τ goes 1→5 d. Physical motive: the slow mode drifts over longer τ, so strict invariance should relax. `λ_inv(τ) = sched(τ)`; `λ_off(τ) = LAMBDA_OFF_BASE · sched(τ)` (keeps the Barlow diagonal/off-diagonal balance while grading both).

## Decisions (Session 36, resolved 2026-06-12)
1. off-diagonal **also** τ-graded — both terms scaled by `sched(τ)`.
2. input = MJO daily-mean `X_MJO_bp20_90` (Lee + Lanczos 20–90 d bandpass).
3. warm-start the **MJO Stage-1 encoder** (`EncoderMJO`, nb21) — *not* nb18c (that is BSISO 2-D; shapes incompatible).
4. small projector sized for D≈7: `64 → 128 → 64 → 7`.
5. new notebook after the NSV chapter (this is nb26).

## Inputs (`MJO/data/processed/` + `MJO/nsv/`)
- `X_MJO_bp20_90.npy` `(N, 3, 1, 180)`, `labels_aligned_mjo_bp20_90.csv` (phase, amplitude, enso_category, weak_mjo, date)
- `MJO/nsv/checkpoints_lag10/encoder_stage1_best.pth` (warm-start)

## ⚠️ Prerequisite
Both the bp20-90 input and the Stage-1 encoder come from the **MJO NSV pipeline (nb21)**, whose full-daily rerun was deferred (Session 35). Refresh them on the full 1979–2023 daily data before trusting these numbers. Cell 2 mtime-checks the input; Cell 4 reports the encoder's date.

## Outputs (`MJO/barlow/`)
`encoder_bt.pth`, `projector_bt.pth`, `embeddings_z7.npy`, `embeddings_f64.npy`, `training_history.json`, `bt_diagnostics.png`, `bt_probe_results.json`, `bt_summary.md`

---

## Cell 1 — Setup + Config

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, time, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

PROJECT_DIR   = '/content/drive/MyDrive/BSISO_SSL_Project'
MJO_DIR       = f'{PROJECT_DIR}/MJO'
PROCESSED_DIR = f'{MJO_DIR}/data/processed'
NSV_CKPT_DIR  = f'{MJO_DIR}/nsv/checkpoints_lag10'
OUT_BASE      = f'{MJO_DIR}/barlow'

# ---- config ----
TAUS            = [1, 2, 3, 4, 5]      # day-lags for the two views
PROJ_DIM        = 3                    # embedding / cross-correlation dim (try 3; was 7 = MJO d_hat)
LATENT_DIM      = 64                   # encoder output (matches nb21)
LAMBDA_OFF_BASE = 5e-3                 # Barlow off-diagonal weight (paper default); tune for small D
BATCH_SIZE      = 256
EPOCHS          = 100
LR              = 1e-3
WEIGHT_DECAY    = 1e-4
WARM_START      = True                 # load nb21 Stage-1 encoder weights
SEED            = 42
VAL_YEARS_STRIDE = 5

# per-dim output folder so D=3 and D=7 runs don't overwrite each other
OUT_DIR = f'{OUT_BASE}/D{PROJ_DIM}'
os.makedirs(OUT_DIR, exist_ok=True)

# ---- tau-grading schedule (Session 44 fix) ----
# STEEP decay so invariance is enforced mainly at tau=1 (where the MJO phase is ~unchanged)
# and barely at tau=5 (where the MJO has propagated). Set LAMBDA_TAU_MIN=0.5, SCHED_MODE='linear'
# to recover the run-1 schedule.
LAMBDA_TAU_MAX = 1.0                   # weight at tau=1
LAMBDA_TAU_MIN = 0.05                  # weight at tau=max
SCHED_MODE     = 'exp'                 # 'exp' (geometric) or 'linear'

def sched(tau):
    f = (tau - min(TAUS)) / (max(TAUS) - min(TAUS))           # 0..1
    if SCHED_MODE == 'exp':
        return LAMBDA_TAU_MAX * (LAMBDA_TAU_MIN / LAMBDA_TAU_MAX) ** f
    return LAMBDA_TAU_MAX + (LAMBDA_TAU_MIN - LAMBDA_TAU_MAX) * f

torch.manual_seed(SEED); np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}   proj_dim D={PROJ_DIM}   taus={TAUS}   out={OUT_DIR}')
print(f'lambda(tau) schedule ({SCHED_MODE}, {LAMBDA_TAU_MAX}->{LAMBDA_TAU_MIN}):',
      {t: round(sched(t), 3) for t in TAUS})
if device.type != 'cuda':
    print('WARNING: no GPU - switch Colab runtime to T4.')

## Cell 2 — Load bp20-90 Data + Freshness Check + Year Split

In [ ]:
X_FILE      = 'X_MJO_bp20_90.npy'
LABELS_FILE = 'labels_aligned_mjo_bp20_90.csv'

X      = np.load(f'{PROCESSED_DIR}/{X_FILE}')
labels = pd.read_csv(f'{PROCESSED_DIR}/{LABELS_FILE}', parse_dates=['date'])
labels['date'] = labels['date'].dt.normalize()
assert X.shape[0] == len(labels), f'X/labels mismatch: {X.shape[0]} vs {len(labels)}'
assert X.shape[1:] == (3, 1, 180), f'unexpected X shape {X.shape[1:]}'

# freshness vs raw OLR (same idea as nb24): warn if the bp20-90 input predates the daily download
RAW_DIR = f'{MJO_DIR}/data/raw'
raw_olr = [f'{RAW_DIR}/{f}' for f in os.listdir(RAW_DIR) if f.startswith('OLR_MJO_') and f.endswith('.nc')] \
          if os.path.isdir(RAW_DIR) else []
if raw_olr:
    dh = (os.path.getmtime(f'{PROCESSED_DIR}/{X_FILE}') - max(os.path.getmtime(f) for f in raw_olr)) / 3600
    print(f'X_bp20_90 mtime - newest raw OLR = {dh:+.1f} h')
    if dh < 0:
        print('  ⚠️ bp20-90 input is OLDER than raw OLR — rerun MJO bp20-90 preprocessing + nb21 on full daily data.')

# squeeze singleton lat → (N, 3, 180); sort by date defensively
order = np.argsort(labels['date'].values)
if not np.array_equal(order, np.arange(len(labels))):
    X = X[order]; labels = labels.iloc[order].reset_index(drop=True)
X = X.reshape(X.shape[0], 3, 180).astype(np.float32)
N = X.shape[0]
dates_all = pd.DatetimeIndex(labels['date'].values)
years_all = dates_all.year.values
print(f'X: {X.shape}  ({dates_all.min().date()} .. {dates_all.max().date()})')

all_years   = sorted(np.unique(years_all).tolist())
val_years   = all_years[::VAL_YEARS_STRIDE]
train_years = sorted(set(all_years) - set(val_years))
is_val_day  = np.isin(years_all, val_years)
print(f'Val years ({len(val_years)}): {val_years}')

Xt = torch.from_numpy(X)   # keep one CPU copy; datasets index into it

## Cell 3 — Multi-τ Pair Construction (same-split, per τ)

For each τ build `(anchor, target)` index pairs with `delta_days == τ` and anchor/target in the **same** split (no train↔val leakage). One train DataLoader per τ; each batch is a single τ so the 7×7 cross-correlation is computed on a uniform-τ batch.

In [ ]:
class PairIdxDataset(Dataset):
    def __init__(self, Xt, idx_t, idx_t1):
        self.Xt = Xt; self.idx_t = idx_t; self.idx_t1 = idx_t1
    def __len__(self): return len(self.idx_t)
    def __getitem__(self, k): return self.Xt[self.idx_t[k]], self.Xt[self.idx_t1[k]]

def build_pairs(tau):
    delta = (dates_all[tau:] - dates_all[:-tau]).days
    same_split = is_val_day[:-tau] == is_val_day[tau:]
    valid = (delta == tau) & same_split
    it  = np.where(valid)[0]
    it1 = it + tau
    return it, it1

train_loaders, val_loaders, pair_counts = {}, {}, {}
for tau in TAUS:
    it, it1 = build_pairs(tau)
    anchor_val = is_val_day[it]
    tr = ~anchor_val; va = anchor_val
    train_loaders[tau] = DataLoader(PairIdxDataset(Xt, it[tr], it1[tr]),
                                    batch_size=BATCH_SIZE, shuffle=True,
                                    num_workers=2, pin_memory=(device.type=='cuda'), drop_last=True)
    val_loaders[tau]   = DataLoader(PairIdxDataset(Xt, it[va], it1[va]),
                                    batch_size=BATCH_SIZE, shuffle=False,
                                    num_workers=2, pin_memory=(device.type=='cuda'))
    pair_counts[tau] = {'train': int(tr.sum()), 'val': int(va.sum())}
    print(f'τ={tau}: {int(tr.sum())} train / {int(va.sum())} val pairs')

## Cell 4 — Encoder (warm-start from nb21) + Projector + Barlow Loss

In [ ]:
class EncoderMJO(nn.Module):
    """1-D CNN encoder (identical to nb21 so the Stage-1 checkpoint loads)."""
    def __init__(self, latent_dim=64):
        super().__init__()
        self.conv1 = nn.Conv1d(3,   16,  4, 2, 1, bias=False); self.bn1 = nn.BatchNorm1d(16)
        self.conv2 = nn.Conv1d(16,  32,  3, 1, 1, bias=False); self.bn2 = nn.BatchNorm1d(32)
        self.conv3 = nn.Conv1d(32,  32,  4, 2, 1, bias=False); self.bn3 = nn.BatchNorm1d(32)
        self.conv4 = nn.Conv1d(32,  64,  3, 1, 1, bias=False); self.bn4 = nn.BatchNorm1d(64)
        self.conv5 = nn.Conv1d(64,  128, 4, 2, 1, bias=False); self.bn5 = nn.BatchNorm1d(128)
        self.gap   = nn.AdaptiveAvgPool1d(1)
        self.fc    = nn.Linear(128, latent_dim)
    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))
        x = F.relu(self.bn4(self.conv4(x)))
        x = F.relu(self.bn5(self.conv5(x)))
        return self.fc(self.gap(x).flatten(1))


class Projector(nn.Module):
    """64 → 128 → 64 → D (BN+ReLU between layers). Small, NSV-sized output."""
    def __init__(self, in_dim=64, out_dim=7):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 128), nn.BatchNorm1d(128), nn.ReLU(inplace=True),
            nn.Linear(128, 64),     nn.BatchNorm1d(64),  nn.ReLU(inplace=True),
            nn.Linear(64, out_dim))
    def forward(self, x): return self.net(x)


encoder   = EncoderMJO(LATENT_DIM).to(device)
projector = Projector(LATENT_DIM, PROJ_DIM).to(device)

ckpt = f'{NSV_CKPT_DIR}/encoder_stage1_best.pth'
if WARM_START and os.path.exists(ckpt):
    encoder.load_state_dict(torch.load(ckpt, map_location=device))
    mt = time.strftime('%Y-%m-%d', time.localtime(os.path.getmtime(ckpt)))
    print(f'Warm-started encoder from nb21 Stage-1 ({mt}).')
else:
    print('Encoder trained from scratch (no warm-start checkpoint found).' if WARM_START else 'Cold-start encoder.')


def barlow_cross_corr(zA, zB, eps=1e-5):
    """Batch-normalized cross-correlation matrix C (D,D) in [-1,1]."""
    zA = (zA - zA.mean(0)) / (zA.std(0) + eps)
    zB = (zB - zB.mean(0)) / (zB.std(0) + eps)
    return (zA.T @ zB) / zA.size(0)

def barlow_loss(C, lam_inv, lam_off):
    on  = torch.diagonal(C).add(-1.0).pow(2).sum()
    off = (C - torch.diag(torch.diagonal(C))).pow(2).sum()
    return lam_inv * on + lam_off * off, on.item(), off.item()

_z = projector(encoder(torch.randn(8, 3, 180).to(device)))
print(f'projector out: {tuple(_z.shape)}  (expect (8, {PROJ_DIM}))')

## Cell 5 — Training (per-τ Barlow loss, τ-graded λ)

Each epoch interleaves one batch from every τ-loader; the batch's loss uses `λ_inv(τ)=sched(τ)` and `λ_off(τ)=LAMBDA_OFF_BASE·sched(τ)`. Per-τ on/off-diagonal means are tracked to monitor how invariance relaxes with τ.

In [ ]:
from itertools import cycle

opt = optim.Adam(list(encoder.parameters()) + list(projector.parameters()),
                 lr=LR, weight_decay=WEIGHT_DECAY)
sch = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=LR * 0.01)

hist = {'epoch_loss': [], 'on_by_tau': {t: [] for t in TAUS}, 'off_by_tau': {t: [] for t in TAUS}}
steps_per_epoch = min(len(train_loaders[t]) for t in TAUS)
best_loss, best_state = float('inf'), None
t0 = time.time()

for ep in range(EPOCHS):
    encoder.train(); projector.train()
    iters = {t: iter(train_loaders[t]) for t in TAUS}
    ep_loss = 0.0; on_acc = {t: 0.0 for t in TAUS}; off_acc = {t: 0.0 for t in TAUS}
    for _ in range(steps_per_epoch):
        opt.zero_grad(); total = 0.0
        for tau in TAUS:
            xa, xb = next(iters[tau])
            xa = xa.to(device, non_blocking=True); xb = xb.to(device, non_blocking=True)
            C = barlow_cross_corr(projector(encoder(xa)), projector(encoder(xb)))
            loss, on, off = barlow_loss(C, sched(tau), LAMBDA_OFF_BASE * sched(tau))
            total = total + loss
            on_acc[tau] += on / PROJ_DIM            # mean (1-C_ii)^2 per dim
            off_acc[tau] += off / (PROJ_DIM * (PROJ_DIM - 1))
        total.backward(); opt.step()
        ep_loss += total.item()
    sch.step()
    hist['epoch_loss'].append(ep_loss / steps_per_epoch)
    for t in TAUS:
        hist['on_by_tau'][t].append(on_acc[t] / steps_per_epoch)
        hist['off_by_tau'][t].append(off_acc[t] / steps_per_epoch)
    if hist['epoch_loss'][-1] < best_loss:
        best_loss = hist['epoch_loss'][-1]
        best_state = (copy.deepcopy(encoder.state_dict()), copy.deepcopy(projector.state_dict()))
    if (ep + 1) % 10 == 0 or ep < 3:
        on1 = hist['on_by_tau'][1][-1]; on5 = hist['on_by_tau'][5][-1]
        print(f'ep {ep+1:3d}/{EPOCHS}  loss={hist["epoch_loss"][-1]:.4f}  '
              f'on(τ1)={on1:.3f} on(τ5)={on5:.3f}  lr={sch.get_last_lr()[0]:.1e}')

if best_state is not None:
    encoder.load_state_dict(best_state[0]); projector.load_state_dict(best_state[1])
torch.save(encoder.state_dict(),   f'{OUT_DIR}/encoder_bt.pth')
torch.save(projector.state_dict(), f'{OUT_DIR}/projector_bt.pth')
with open(f'{OUT_DIR}/training_history.json', 'w') as f:
    json.dump(hist, f, indent=2)
print(f'\nDone in {(time.time()-t0)/60:.1f} min. best loss={best_loss:.4f}')

## Cell 6 — Diagnostics: per-τ Invariance Trajectory

On-diagonal (1−C_ii)² should fall toward 0; off-diagonal C_ij² toward 0. Under the τ-graded loss we expect **invariance to relax with τ** — longer-τ pairs settle at a *higher* residual on-diagonal (the mode drifted), which is the slow-evolution signal we set out to monitor.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(hist['epoch_loss'], lw=2); axes[0].set_title('Total τ-graded Barlow loss', fontweight='bold')
axes[0].set_xlabel('epoch'); axes[0].set_ylabel('loss'); axes[0].grid(alpha=0.3)

cmap = plt.cm.viridis(np.linspace(0, 0.9, len(TAUS)))
for c, t in zip(cmap, TAUS):
    axes[1].plot(hist['on_by_tau'][t], color=c, lw=2, label=f'τ={t}')
    axes[2].plot(hist['off_by_tau'][t], color=c, lw=2, label=f'τ={t}')
axes[1].set_title('On-diagonal mean (1−C_ii)²  (invariance)', fontweight='bold')
axes[2].set_title('Off-diagonal mean C_ij²  (redundancy)', fontweight='bold')
for ax in axes[1:]:
    ax.set_xlabel('epoch'); ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/bt_diagnostics.png', dpi=140, bbox_inches='tight'); plt.show()

print('Final on-diagonal (1−C_ii)² by τ (higher = more drift / relaxed invariance):')
for t in TAUS:
    print(f'  τ={t}: on={hist["on_by_tau"][t][-1]:.4f}  off={hist["off_by_tau"][t][-1]:.4f}')

## Cell 7 — Extract Embeddings + Linear Probes + ENSO Displacement

Encode every day → 7-D projector embedding `z7` (NSV-matched, primary) and 64-D encoder feature `f64` (BT convention). Probe RMM phase (8-class) and ENSO (balanced) on active-MJO days; ENSO displacement z-score on `z7`. Compare to nb14 sup z=12.21 / nb23 NSV v-space z=20.88.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score

encoder.eval(); projector.eval()
f64 = np.zeros((N, LATENT_DIM), np.float32); z7 = np.zeros((N, PROJ_DIM), np.float32)
with torch.no_grad():
    for s in range(0, N, 512):
        b = Xt[s:s+512].to(device)
        fe = encoder(b); f64[s:s+512] = fe.cpu().numpy(); z7[s:s+512] = projector(fe).cpu().numpy()
np.save(f'{OUT_DIR}/embeddings_z7.npy', z7); np.save(f'{OUT_DIR}/embeddings_f64.npy', f64)

phase = labels['phase'].values.astype(int)
amp   = labels['amplitude'].values
enso  = labels['enso_category'].values
weak  = labels['weak_mjo'].values.astype(bool)
active = (~weak) & (amp >= 1.0)
tr = (~is_val_day) & active; va = is_val_day & active

def probe(emb):
    clf = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
    clf.fit(emb[tr], phase[tr]); ph = float(accuracy_score(phase[va], clf.predict(emb[va])))
    clfb = LogisticRegression(max_iter=1000, C=1.0, class_weight='balanced', random_state=42)
    clfb.fit(emb[tr], enso[tr]); en = float(balanced_accuracy_score(enso[va], clfb.predict(emb[va])))
    return ph, en

ph7, en7 = probe(z7); ph64, en64 = probe(f64)

def enso_z(emb):
    idx = np.where(active)[0]; e = emb[idx]; lab = labels.iloc[idx].reset_index(drop=True)
    obs = []
    for p in range(1, 9):
        mEN = (lab['phase']==p) & (lab['enso_category']=='El Nino')
        mLN = (lab['phase']==p) & (lab['enso_category']=='La Nina')
        if mEN.sum()<3 or mLN.sum()<3: continue
        obs.append(np.linalg.norm(e[mEN.values].mean(0) - e[mLN.values].mean(0)))
    rng = np.random.default_rng(42); base = []
    for _ in range(100):
        sh = lab['enso_category'].sample(frac=1, random_state=rng.integers(1e6)).values; t = []
        for p in range(1, 9):
            mph = (lab['phase']==p).values
            mEN = mph & (sh=='El Nino'); mLN = mph & (sh=='La Nina')
            if mEN.sum()<3 or mLN.sum()<3: continue
            t.append(np.linalg.norm(e[mEN].mean(0) - e[mLN].mean(0)))
        if t: base.append(np.mean(t))
    return float((np.mean(obs) - np.mean(base)) / (np.std(base) + 1e-8))

z7_enso = enso_z(z7)
print('=' * 60)
print(f'7-D projector:  RMM phase {ph7*100:.1f}%   ENSO bal {en7*100:.1f}%   ENSO z {z7_enso:.2f}')
print(f'64-D encoder:   RMM phase {ph64*100:.1f}%  ENSO bal {en64*100:.1f}%')
print(f'(refs: nb14 sup z=12.21, nb23 NSV v-space z=20.88; random phase 12.5%, ENSO 33.3%)')
print('=' * 60)

res = {'proj_dim': PROJ_DIM, 'taus': TAUS, 'lambda_off_base': LAMBDA_OFF_BASE,
       'warm_start': WARM_START, 'pair_counts': pair_counts,
       'z7': {'phase_val': ph7, 'enso_bal': en7, 'enso_z': z7_enso},
       'f64': {'phase_val': ph64, 'enso_bal': en64},
       'on_diag_final': {t: hist['on_by_tau'][t][-1] for t in TAUS},
       'off_diag_final': {t: hist['off_by_tau'][t][-1] for t in TAUS}}
with open(f'{OUT_DIR}/bt_probe_results.json', 'w') as f:
    json.dump(res, f, indent=2)

summary = f"""# MJO Barlow Twins (τ-graded, D={PROJ_DIM}) — Summary

Warm-start: {WARM_START} (nb21 Stage-1).  τ={TAUS}, λ_off_base={LAMBDA_OFF_BASE}.

| Representation | RMM phase val | ENSO bal | ENSO z |
|---|---|---|---|
| 7-D projector | {ph7*100:.1f}% | {en7*100:.1f}% | {z7_enso:.2f} |
| 64-D encoder  | {ph64*100:.1f}% | {en64*100:.1f}% | — |

Refs: nb14 sup z=12.21, nb23 NSV v-space z=20.88. Random phase 12.5%, ENSO 33.3%.

Per-τ final on-diagonal (1−C_ii)²: {{ {', '.join(f'{t}:{hist["on_by_tau"][t][-1]:.3f}' for t in TAUS)} }}
(rising with τ = invariance relaxing as the mode drifts — the intended τ-grading effect).
"""
with open(f'{OUT_DIR}/bt_summary.md', 'w') as f:
    f.write(summary)
print('Saved embeddings + probe results + summary to MJO/barlow/')

## Cell 8 — Modulation Dimensionality: How Many Dims Carry the ENSO Signal

The BT projector output is **whitened by construction** (C→I ⇒ ~uniform per-dim variance), so raw variance% is uninformative for z7. Instead we decompose the **per-phase EN−LN displacement matrix** `M (8×D)` by SVD: the singular-value spectrum shows how many independent directions the ENSO modulation occupies, and the **participation ratio** `(Σλ)²/Σλ²` gives the *effective number* of modulation dimensions. Reported for both z7 (projector) and f64 (encoder), plus per-dim ENSO effect size.

In [ ]:
from numpy.linalg import svd

# How many dimensions carry the ENSO modulation, and what % each occupies?
# NOTE: the BT projector output is whitened by construction (C->I => ~uniform per-dim
# variance), so raw variance% is uninformative for z7. The meaningful decomposition is
# the SINGULAR SPECTRUM of the per-phase EN-LN displacement matrix M (n_phase x D):
# its singular values say how many independent directions the ENSO modulation occupies.

idx = np.where(active)[0]
lab = labels.iloc[idx].reset_index(drop=True)

def modulation_spectrum(emb, name):
    e = emb[idx]
    ec = e - e.mean(0)
    s = svd(ec, full_matrices=False, compute_uv=False)
    pca_var = (s ** 2) / (s ** 2).sum()
    rows = []
    for p in range(1, 9):
        mEN = ((lab['phase'] == p) & (lab['enso_category'] == 'El Nino')).values
        mLN = ((lab['phase'] == p) & (lab['enso_category'] == 'La Nina')).values
        if mEN.sum() < 3 or mLN.sum() < 3:
            continue
        rows.append(e[mEN].mean(0) - e[mLN].mean(0))
    M = np.array(rows)
    sm = svd(M, full_matrices=False, compute_uv=False)
    mod_var = (sm ** 2) / (sm ** 2).sum()
    pr = float(mod_var.sum() ** 2 / (mod_var ** 2).sum())
    enso_v = lab['enso_category'].values
    mEN = enso_v == 'El Nino'; mLN = enso_v == 'La Nina'
    eff = np.abs(e[mEN].mean(0) - e[mLN].mean(0)) / (e.std(0) + 1e-8)
    print(f'\\n[{name}]  D={e.shape[1]}')
    print(f'  embedding PCA var%   (top8): {(pca_var[:8]*100).round(1)}')
    print(f'  ENSO-modulation sv%  (top8): {(mod_var[:8]*100).round(1)}')
    print(f'  effective # modulation dims (participation ratio): {pr:.2f}')
    print(f'  per-dim ENSO effect size (sorted desc, top8): {np.sort(eff)[::-1][:8].round(2)}')
    return pca_var, mod_var, eff, pr

pca7,  mod7,  eff7,  pr7  = modulation_spectrum(z7,  'z7 projector')
pca64, mod64, eff64, pr64 = modulation_spectrum(f64, 'f64 encoder')

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
axes[0].bar(np.arange(1, len(mod7) + 1), mod7 * 100, color='tab:red', alpha=0.85)
axes[0].set_title(f'z7: ENSO-modulation singular spectrum\\n(eff dims = {pr7:.2f})', fontweight='bold')
axes[0].set_xlabel('modulation direction'); axes[0].set_ylabel('% of modulation energy')
axes[1].bar(np.arange(1, min(15, len(mod64)) + 1), (mod64 * 100)[:15], color='tab:purple', alpha=0.85)
axes[1].set_title(f'f64: ENSO-modulation singular spectrum\\n(eff dims = {pr64:.2f})', fontweight='bold')
axes[1].set_xlabel('modulation direction (top 15)'); axes[1].set_ylabel('% of modulation energy')
axes[2].bar(np.arange(1, len(eff7) + 1), np.sort(eff7)[::-1], color='tab:red', alpha=0.6)
axes[2].set_title('z7: per-dim ENSO effect size (sorted)', fontweight='bold')
axes[2].set_xlabel('dim rank'); axes[2].set_ylabel('|EN-LN| / std')
for ax in axes: ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/bt_modulation_dims.png', dpi=140, bbox_inches='tight'); plt.show()

with open(f'{OUT_DIR}/bt_modulation_dims.json', 'w') as f:
    json.dump({'z7':  {'eff_mod_dims': pr7,  'mod_sv_pct': (mod7 * 100).round(2).tolist(),
                       'pca_var_pct': (pca7 * 100).round(2).tolist()},
               'f64': {'eff_mod_dims': pr64, 'mod_sv_pct': (mod64 * 100).round(2).tolist()[:15]}}, f, indent=2)
print('\\nSaved bt_modulation_dims.png + .json')
print('Interpretation: \"eff dims\" = how many independent directions the ENSO modulation occupies.')

## Cell 9 — Visualize the 7-D SSL Embedding (PCA + t-SNE)

Project z7 to 2-D (PCA and t-SNE) on active-MJO days, colored by RMM phase, ENSO, and amplitude. **Reading guide:** a phase-colored *loop/ring* (phases ordered around it) means the MJO cycle is retained; a clean *El Niño/La Niña split* means ENSO is captured; an amplitude gradient from center outward means the radius encodes intensity. In run 1 (gentle τ-grading) phase was random — expect a phase loop to emerge after the steep-τ-decay re-run.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# Visualize the SSL embedding (z7 = projector output, dim=PROJ_DIM) in 2-D via PCA and
# t-SNE, on active-MJO days. Subsample for t-SNE speed. Color by RMM phase (does it trace
# the MJO cycle as a loop?), ENSO (EN/LN split?), and amplitude (radial structure?).
idxa = np.where(active)[0]
rng = np.random.default_rng(0)
sub = rng.choice(idxa, size=min(5000, len(idxa)), replace=False)
Zs = z7[sub]
ph = labels['phase'].values[sub].astype(int)
en = labels['enso_category'].values[sub]
am = labels['amplitude'].values[sub]

pca2  = PCA(2).fit_transform(Zs - Zs.mean(0))
tsne2 = TSNE(n_components=2, perplexity=30, init='pca', learning_rate='auto',
             random_state=0).fit_transform(Zs)

phase_colors = plt.cm.hsv(np.linspace(0, 1, 9))   # cyclic colormap for phase (loop check)
enso_pal = {'El Nino': '#d62728', 'Neutral': '#7f7f7f', 'La Nina': '#1f77b4'}

def scatter_by(ax, XY, kind):
    if kind == 'phase':
        for p in range(1, 9):
            m = ph == p
            ax.scatter(XY[m, 0], XY[m, 1], s=6, alpha=0.6, color=phase_colors[p-1], label=f'P{p}')
        ax.set_title('by RMM phase'); ax.legend(fontsize=6, ncol=2, markerscale=2)
    elif kind == 'enso':
        for c in ['El Nino', 'Neutral', 'La Nina']:
            m = en == c
            ax.scatter(XY[m, 0], XY[m, 1], s=6, alpha=0.5, color=enso_pal[c], label=c)
        ax.set_title('by ENSO'); ax.legend(fontsize=7, markerscale=2)
    else:
        sc = ax.scatter(XY[:, 0], XY[:, 1], s=6, alpha=0.6, c=am, cmap='viridis')
        ax.set_title('by RMM amplitude'); plt.colorbar(sc, ax=ax, fraction=0.046)
    ax.set_xticks([]); ax.set_yticks([])

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
for j, kind in enumerate(['phase', 'enso', 'amplitude']):
    scatter_by(axes[0, j], pca2,  kind)
    scatter_by(axes[1, j], tsne2, kind)
axes[0, 0].set_ylabel('PCA-2D', fontsize=12)
axes[1, 0].set_ylabel('t-SNE-2D', fontsize=12)
plt.suptitle(f'{PROJ_DIM}-D Barlow-Twins embedding (projector) -> 2-D  (n={len(sub)} active days)',
             fontsize=14, fontweight='bold', y=0.99)
plt.tight_layout(rect=[0, 0, 1, 0.98])
p = f'{OUT_DIR}/bt_embedding_2d.png'
plt.savefig(p, dpi=130, bbox_inches='tight'); plt.show(); print('Saved', p)
print('Read: a phase-colored LOOP = MJO cycle retained; an EN/LN split = ENSO captured.')

---
## Done!

**Send back:** `bt_diagnostics.png`, the Cell-7 printout (7-D phase / ENSO bal / ENSO z), and `bt_summary.md`.

**How to read it:**
- **Per-τ on-diagonal trajectory (Cell 6):** should rise with τ — invariance relaxing as the slow mode drifts. That's the τ-grading working as designed.
- **7-D phase / ENSO z (Cell 7):** the headline. If z7 approaches the NSV v-space z=20.88 with no labels used, Barlow Twins recovered the ENSO-modulated MJO structure self-supervised.
- **Knobs if collapse / weak signal:** `LAMBDA_OFF_BASE` (raise toward ~0.05–0.1 for small D if dims correlate), `WARM_START`, `BATCH_SIZE`.

**Reminder:** refresh `X_MJO_bp20_90.npy` + nb21 Stage-1 encoder on the full 1979–2023 daily data first (Cell 2 warns if stale).

---
*DDCS Project | jh9141@nyu.edu*